# Stage 6b — CPI Leaderboards with Uncertainty

Stage 6 ranks every qualified player by the **Clutch Performance Index**, CPI = (56 / n_p) · Σ V, where V = ΔP_scores · L⁺ − ΔP_concedes · L⁻ and 56 is the median number of actions per appearance. This notebook adds what a ranking needs before it can be published:

1. **Per-position leaderboards.** Goalkeepers act almost only in the conceding channel, so they are ranked on their own.
2. **Uncertainty.** Whole matches are resampled (1,000 draws). Each player gets a 95% interval for CPI, a median rank within position, and the probability of being in the position's top 10.
3. **What the stakes change.** Each player's rank by CPI is compared with their rank by plain VAEP per 56 actions (the same actions, no stakes). The largest risers show who the weighting rewards.
4. **Holding a lead.** The conceding-channel part of CPI, 56 / n_p · Σ(−ΔP_concedes · L⁻), rates how much goal-prevention value a player adds when conceding is costly.

The qualified set (n_p ≥ 700, as in Stage 6) is fixed before resampling, so every draw ranks the same players.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from scipy.stats import spearmanr

# Repository root (the notebooks live one level below it)
BASE_DIR = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')

OUT_DIR = BASE_DIR / 'stage6_cpi' / 'leaderboards'
OUT_DIR.mkdir(parents=True, exist_ok=True)

M, MIN_ACTIONS, B, TOP, SEED = 56, 700, 1000, 10, 42
POSITIONS = ['forward', 'midfielder', 'defender', 'goalkeeper']
pd.set_option('display.width', 220)
print(f'Project folder: {BASE_DIR.resolve().name}')

Project folder: clutch-cpi-soccer


In [2]:
cv = pd.read_parquet(BASE_DIR / 'data' / 'clutch_values_signed.parquet',
                     columns=['game_id', 'player_id', 'V', 'vaep_value', 'dP_concedes', 'L_minus'])
cv['hold'] = -cv['dP_concedes'] * cv['L_minus']          # conceding-channel part of V

# Per player and match: the sums every statistic below is built from
pg = (cv.groupby(['player_id', 'game_id'])
        .agg(n=('V', 'size'), V=('V', 'sum'), vaep=('vaep_value', 'sum'), hold=('hold', 'sum'))
        .reset_index())
del cv

info = pd.read_csv(BASE_DIR / 'stage6_cpi' / 'ranking_full.csv',
                   usecols=['player_id', 'name', 'team', 'league', 'position_group', 'goals', 'CPI', 'qualified'])
info = info[info['qualified']].set_index('player_id')
pg = pg[pg['player_id'].isin(info.index)]

players = info.index.values
games   = np.sort(pg['game_id'].unique())
pi = pd.Series(np.arange(len(players)), index=players).loc[pg['player_id']].values
gi = pd.Series(np.arange(len(games)), index=games).loc[pg['game_id']].values
mat = {k: sparse.csr_matrix((pg[k].values.astype(float), (pi, gi)), shape=(len(players), len(games)))
       for k in ['n', 'V', 'vaep', 'hold']}


def ratings(w):
    tot = {k: m @ w for k, m in mat.items()}
    return pd.DataFrame({'CPI': M * tot['V'] / tot['n'], 'VAEP': M * tot['vaep'] / tot['n'],
                         'HOLD': M * tot['hold'] / tot['n']}, index=players)


obs = ratings(np.ones(len(games)))
assert np.allclose(obs['CPI'], info.loc[players, 'CPI'], atol=1e-9), 'CPI does not match Stage 6'
print(f'{len(players):,} qualified players, {len(games):,} matches; CPI reproduces Stage 6 exactly')

1,589 qualified players, 1,826 matches; CPI reproduces Stage 6 exactly


In [3]:
pos = info.loc[players, 'position_group'].values
rng = np.random.default_rng(SEED)
draws = {k: np.empty((B, len(players))) for k in ['CPI', 'HOLD']}
ranks = {k: np.empty((B, len(players))) for k in ['CPI', 'HOLD']}
for b in range(B):
    w = np.bincount(rng.integers(0, len(games), len(games)), minlength=len(games)).astype(float)
    r = ratings(w)
    for k in draws:
        draws[k][b] = r[k].values
        ranks[k][b] = r[k].groupby(pos).rank(ascending=False).values

def summary(k):
    s = pd.DataFrame({k: obs[k], 'ci_low': np.nanpercentile(draws[k], 2.5, axis=0),
                      'ci_high': np.nanpercentile(draws[k], 97.5, axis=0),
                      'median_rank': np.nanmedian(ranks[k], axis=0),
                      f'P(top {TOP})': (ranks[k] <= TOP).mean(axis=0)}, index=players)
    s['rank'] = obs[k].groupby(pos).rank(ascending=False).values
    return info[['name', 'team', 'league', 'position_group', 'goals']].join(s)

cpi_board, hold_board = summary('CPI'), summary('HOLD')
print('bootstrap done:', B, 'match resamples')

bootstrap done: 1000 match resamples


In [4]:
def show(board, k, position, n=5):
    t = board[board['position_group'] == position].sort_values('rank').head(n)
    t = t.assign(interval=[f'[{a:.3f}, {b:.3f}]' for a, b in zip(t['ci_low'], t['ci_high'])])
    return t[['rank', 'name', 'team', k, 'interval', 'median_rank', f'P(top {TOP})']].round(3)

for p in ['forward', 'midfielder', 'defender']:
    print(f'\n=== CPI leaderboard: {p}s (of {int((pos == p).sum())}) ===')
    print(show(cpi_board, 'CPI', p).to_string(index=False))
print(f"\n=== Holding a lead: goalkeepers (of {int((pos == 'goalkeeper').sum())}), conceding-channel CPI ===")
print(show(hold_board, 'HOLD', 'goalkeeper').to_string(index=False))
for p in ['defender', 'midfielder']:
    print(f'\n=== Holding a lead: {p}s ===')
    print(show(hold_board, 'HOLD', p).to_string(index=False))

cpi_board.sort_values(['position_group', 'rank']).to_csv(OUT_DIR / 'cpi_leaderboard_by_position.csv')
hold_board.sort_values(['position_group', 'rank']).to_csv(OUT_DIR / 'holding_a_lead_leaderboard_by_position.csv')


=== CPI leaderboard: forwards (of 287) ===
 rank        name           team   CPI       interval  median_rank  P(top 10)
  1.0   M. Icardi Internazionale 0.618 [0.220, 1.065]          2.0      0.831
  2.0   E. Cavani            PSG 0.497 [0.263, 0.716]          5.0      0.746
  3.0 C. Immobile          Lazio 0.430 [0.158, 0.705]         10.0      0.529
  4.0 A. Kramarić     Hoffenheim 0.413 [0.216, 0.633]         12.0      0.431
  5.0    J. Vardy Leicester City 0.382 [0.145, 0.645]         15.0      0.369

=== CPI leaderboard: midfielders (of 598) ===
 rank            name            team   CPI       interval  median_rank  P(top 10)
  1.0 F. Bernardeschi        Juventus 0.318 [0.112, 0.545]         10.0      0.523
  2.0 Kwon Chang-Hoon           Dijon 0.313 [0.133, 0.527]          9.0      0.558
  3.0      Rony Lopes          Monaco 0.299 [0.149, 0.456]         12.0      0.468
  4.0       W. Khazri          Rennes 0.298 [0.116, 0.514]         12.0      0.463
  5.0       R. Fraser AFC 

In [5]:
mv = info[['name', 'team', 'position_group', 'goals']].join(obs)
mv['rank_CPI']  = mv.groupby('position_group')['CPI'].rank(ascending=False)
mv['rank_VAEP'] = mv.groupby('position_group')['VAEP'].rank(ascending=False)
mv['size'] = mv.groupby('position_group')['CPI'].transform('size')
mv['rise_pct'] = 100 * (mv['rank_VAEP'] - mv['rank_CPI']) / mv['size']   # percentile places gained

print('How much do the stakes change the order? Spearman rho, CPI vs plain VAEP, within position:')
for p in POSITIONS:
    g = mv[mv['position_group'] == p]
    print(f"  {p:<11} {spearmanr(g['CPI'], g['VAEP']).statistic:.3f}  ({len(g)} players)")
for p in ['defender', 'midfielder', 'forward']:
    t = mv[(mv['position_group'] == p) & (mv['rank_CPI'] <= 50)].nlargest(5, 'rise_pct')
    print(f'\nBiggest risers into the top 50 {p}s (rank by plain VAEP -> rank by CPI):')
    print(t[['name', 'team', 'rank_VAEP', 'rank_CPI', 'rise_pct']].round(1).to_string(index=False))
mv.to_csv(OUT_DIR / 'cpi_vs_vaep_movers.csv')

How much do the stakes change the order? Spearman rho, CPI vs plain VAEP, within position:
  forward     0.909  (287 players)
  midfielder  0.853  (598 players)
  defender    0.865  (609 players)
  goalkeeper  0.782  (95 players)

Biggest risers into the top 50 defenders (rank by plain VAEP -> rank by CPI):
           name           team  rank_VAEP  rank_CPI  rise_pct
L. De Silvestri         Torino      154.0      37.0      19.2
       M. Sakho Crystal Palace      133.0      19.0      18.7
   F. Eboa Eboa       Guingamp      120.0      48.0      11.8
     Lucas Lima         Nantes       99.0      28.0      11.7
      K. Rausch           Köln       97.0      35.0      10.2

Biggest risers into the top 50 midfielders (rank by plain VAEP -> rank by CPI):
         name           team  rank_VAEP  rank_CPI  rise_pct
 Z. Junuzović  Werder Bremen      171.0      43.0      21.4
      W. Zaha Crystal Palace      132.0      46.0      14.4
 A. Halilović     Las Palmas       92.0      21.0      11.

In [6]:
# How settled is each top 5? Share of the observed top 5 that the bootstrap keeps in the top 10.
rows = []
for k, board in [('CPI', cpi_board), ('HOLD', hold_board)]:
    for p in POSITIONS:
        t = board[board['position_group'] == p].sort_values('rank').head(5)
        rows.append({'board': k, 'position': p, 'mean P(top 10) of top 5': t[f'P(top {TOP})'].mean(),
                     'widest interval / value': ((t['ci_high'] - t['ci_low']) / t[k].abs()).max()})
print(pd.DataFrame(rows).round(2).to_string(index=False))

board   position  mean P(top 10) of top 5  widest interval / value
  CPI    forward                     0.58                     1.37
  CPI midfielder                     0.50                     1.49
  CPI   defender                     0.55                     1.63
  CPI goalkeeper                     0.73                    48.56
 HOLD    forward                     0.62                     3.99
 HOLD midfielder                     0.51                     2.33
 HOLD   defender                     0.63                     1.98
 HOLD goalkeeper                     0.66                     1.33
